In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [2]:
data3 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/ENEMDU2/2019/BDD_ENEMDU_2019_03_SPSS/BDD_ENEMDU_2019_03_SPSS/201903_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
data6 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/ENEMDU2/2019/BDD_ENEMDU_2019_06_SPSS/BDD_ENEMDU_2019_06_SPSS/201906_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
data9 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/ENEMDU2/2019/BDD_ENEMDU_2019_09_SPSS/BDD_ENEMDU_2019_09_SPSS/enemdu_personas_2019_09.sav", convert_categoricals=False) # para bases de stata
data12 = pd.read_spss(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/ENEMDU2/2019/BDD_ENEMDU_2019_12_SPSS/enemdu_persona_201912.sav", convert_categoricals=False) # para bases de stata

#data3 = pd.read_spss(r"C:\Users\oscarj\OneDrive - Inter-American Development Bank Group\Desktop\ecu\2019\BDD_ENEMDU_2019_03_SPSS\BDD_ENEMDU_2019_03_SPSS\201903_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
#data6 = pd.read_spss(r"C:\Users\oscarj\OneDrive - Inter-American Development Bank Group\Desktop\ecu\2019\BDD_ENEMDU_2019_06_SPSS\BDD_ENEMDU_2019_06_SPSS\201906_EnemduBDD_15anios.sav", convert_categoricals=False) # para bases de stata
#data9 = pd.read_spss(r"C:\Users\oscarj\OneDrive - Inter-American Development Bank Group\Desktop\ecu\2019\BDD_ENEMDU_2019_09_SPSS\BDD_ENEMDU_2019_09_SPSS\enemdu_personas_2019_09.sav", convert_categoricals=False) # para bases de stata
#data12 = pd.read_spss(r"C:\Users\oscarj\OneDrive - Inter-American Development Bank Group\Desktop\ecu\2019\BDD_ENEMDU_2019_12_SPSS\enemdu_persona_201912.sav", convert_categoricals=False) # para bases de stata

In [3]:
data3.columns = [x.lower() for x in data3.columns]
data6.columns = [x.lower() for x in data6.columns]
data9.columns = [x.lower() for x in data9.columns]
data12.columns = [x.lower() for x in data12.columns]

## Revisar los datos

| marzo | junio | septiembre | diciembre |
|-----------|-----------|-----------|-----------|
| area  | area  | area  | area  |
| p15ab | p15ab | ciudad | ciudad  |
|   |   | panelm | panelm  |
| vivienda  | vivienda  | vivienda  | vivienda  |
| hogar  | hogar  | hogar  | hogar  |
| p02  | p02  | p02  | p02  |
| p03  | p03  | p03  | p03  |
| p66  | p66  | p66  | p66  |
| fexp  | fexp  | fexp  | fexp  |
| p20  | p20  | p20  | p20  |
| id_hogar |  id_hogar | id_hogar  | id_hogar  |

En esta encuesta tenemos separadas cuatro diferentes bases para cada trimestre, esto cambia la lógica que habíamos tenido hasta ahora así que de aquí en adelante cambiamos algo del código, mantenemos de acuerdo a las etiquetas de las variables pe63 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

En ests encuesta de 2018, los dos primeros trimestres no tienen la variable de ciudad, zona y sector en este caso utilizaremos la variable p15ab, el lugar donde nació como proxy de región y para asignar los IPC y agregamos una variable de id de hogar propia de la encuesta

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, estas variables las usamos antes para identificar la condición de trabajo para diferentes meses en encuestas anuales o incompletas donde asumíamos que mantenía el mismo salario si estaba ocupado en ese mes, sin mbargo estas variables tenían el problema de no corresponder de forma exacta con el año o mes de la encuesta. Ahora sin embargo podemos cambiar las suposiciones y solamente asumir que si la variable 'trabajando' que pregunta si el individuo trabajó la semana pasada se cumple vamos a asumir que trabajo durante todo el trimestre, de esta manera podemos mejorar las suposiciones de ocupación mensual, mantenemos la idea de que si el individuo trabajo recibe su ingreso laboral reportado.

In [7]:
columnas = pd.Index(['area', 'ciudad', 'p15ab', 'panelm',
            'vivienda', 'hogar', 'p66',
            'fexp', 'p02', 'p03', 'p20', 'id_hogar'])

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [8]:
data3 = data3[columnas.intersection(data3.columns)]
data6 = data6[columnas.intersection(data6.columns)]
data9 = data9[columnas.intersection(data9.columns)]
data12 = data12[columnas.intersection(data12.columns)]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado y limpiamos según los valores de ingrl, para mantener ambas variables para cada base consistente

In [9]:
data12['p66'].value_counts().sort_index(ascending=False)

p66
999999.0     80
9000.0        2
8000.0        1
7650.0        1
7000.0        1
           ... 
16.0          1
15.0          3
10.0          2
4.0           1
0.0         197
Name: count, Length: 630, dtype: int64

In [10]:
data3['p66'] = pd.to_numeric(data3['p66'], errors='coerce')
data3['p66'] = data3['p66'].apply(lambda x: np.nan if x > 8000 else x)
data3['p66'] = data3['p66'].apply(lambda x: np.nan if x < 0 else x)

data6['p66'] = pd.to_numeric(data6['p66'], errors='coerce')
data6['p66'] = data6['p66'].apply(lambda x: np.nan if x > 7200 else x)
data6['p66'] = data6['p66'].apply(lambda x: np.nan if x < 0 else x)

data9['p66'] = pd.to_numeric(data9['p66'], errors='coerce')
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x > 9500 else x)
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x < 0 else x)

data12['p66'] = pd.to_numeric(data12['p66'], errors='coerce')
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x > 9000 else x)
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x < 0 else x)

/tmp/ipykernel_84268/921270504.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['p66'] = pd.to_numeric(data3['p66'], errors='coerce')
/tmp/ipykernel_84268/921270504.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['p66'] = data3['p66'].apply(lambda x: np.nan if x > 8000 else x)
/tmp/ipykernel_84268/921270504.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docume

In [11]:
data3['ingr'] = data3['p66']
data6['ingr'] = data6['p66']
data9['ingr'] = data9['p66']
data12['ingr'] = data12['p66']

/tmp/ipykernel_84268/1418744889.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ingr'] = data3['p66']


Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados la semana pasada, de acuerdo a la variable 'trabajo'

In [12]:
data3['ingr_t1'] = data3.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

data6['ingr_t2'] = data6.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

data9['ingr_t3'] = data9.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

data12['ingr_t4'] = data12.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

/tmp/ipykernel_84268/144387567.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ingr_t1'] = data3.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)


## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [13]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2019]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [14]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [18]:
print(data3['p15ab'][0])
print(data6['p15ab'][0])
print(data9['ciudad'][0])
print(data12['ciudad'][0])

10157.0
250.0
010150
10150.0


In [20]:
print(data3['p15ab'][0])
fac3 = data3['p15ab'].apply(lambda x: len(str(x))).min()
print(fac3)

print(data6['p15ab'][0])
fac6 = data6['p15ab'].apply(lambda x: len(str(x))).min()
print(fac6)

print(data9['ciudad'][0])
fac9 = data9['ciudad'].apply(lambda x: len(str(x))).min()
print(fac9)

print(data12['ciudad'][0])
fac12 = data12['ciudad'].apply(lambda x: len(str(x))).min()
print(fac12)

10157.0
3
250.0
3
010150
6
10150.0
7


In [21]:
# Corregimos los códigos para usarlos cómo texto
data3['ciudad'] = data3['p15ab'].apply(str)
data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == (fac3+4) else x)
data3['ciudad_2'] = data3['ciudad'].apply(lambda x: x[:4])

data6['ciudad'] = data6['p15ab'].apply(str)
data6['ciudad'] = data6['ciudad'].apply(lambda x: '0' + x if len(x) == (fac6+4) else x)
data6['ciudad_2'] = data6['ciudad'].apply(lambda x: x[:4])

data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == (fac9-1) else x)
data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:4])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == fac12 else x)
data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:4])

/tmp/ipykernel_84268/1609601349.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ciudad'] = data3['p15ab'].apply(str)
/tmp/ipykernel_84268/1609601349.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == (fac3+4) else x)
/tmp/ipykernel_84268/1609601349.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the do

Diccionario ciudades disponibles

In [25]:
parroquia_dict = {
    '0101': 'Cuenca',
    '0901': 'Guayaquil',
    '0801': 'Esmeraldas',
    '0701': 'Machala',
    '1308': 'Manta',
    '1701': 'Quito',
    '1101': 'Loja',
    '1801': 'Ambato'
}

def get_parroquia(codigo):
    if codigo in parroquia_dict:
        return parroquia_dict[codigo]
    elif codigo[:2] in ['01', '02', '03', '04', '05', '06', '10', '11', '17', '18']:
        return 'Sierra'
    elif codigo[:2] in ['07', '08', '09', '12', '13', '23', '24']:
        return 'Costa'
    else:
        return 'Nacional'

data3['ciudad_asignada'] = data3['ciudad_2'].apply(get_parroquia)
data6['ciudad_asignada'] = data6['ciudad_2'].apply(get_parroquia)
data9['ciudad_asignada'] = data9['ciudad_2'].apply(get_parroquia)
data12['ciudad_asignada'] = data12['ciudad_2'].apply(get_parroquia)

/tmp/ipykernel_84268/145694457.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ciudad_asignada'] = data3['ciudad_2'].apply(get_parroquia)


In [30]:
data12['ciudad_asignada'].value_counts()

ciudad_asignada
Costa         16730
Sierra        11831
Quito          6453
Guayaquil      5741
Nacional       5686
Ambato         4143
Cuenca         3939
Machala        2895
Esmeraldas      805
Loja            604
Manta           381
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [31]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [32]:
data3['ipc_t1'] = data3.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data3['ipc_base_t1'] = data3.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data6['ipc_t2'] = data6.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data6['ipc_base_t2'] = data6.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data9['ipc_t3'] = data9.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data9['ipc_base_t3'] = data9.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data12['ipc_t4'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data12['ipc_base_t4'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

/tmp/ipykernel_84268/1489697781.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ipc_t1'] = data3.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
/tmp/ipykernel_84268/1489697781.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ipc_base_t1'] = data3.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)


In [33]:
# Calculamos el deflactor
data3['def_t1'] = (data3['ipc_base_t1'] / data3['ipc_t1'])
data6['def_t2'] = (data6['ipc_base_t2'] / data6['ipc_t2'])
data9['def_t3'] = (data9['ipc_base_t3'] / data9['ipc_t3'])
data12['def_t4'] = (data12['ipc_base_t4'] / data12['ipc_t4'])

/tmp/ipykernel_84268/4102172476.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['def_t1'] = (data3['ipc_base_t1'] / data3['ipc_t1'])


Ingreso promedio en el trimeste

In [43]:
data3['ingr_t1_r'] = data3['ingr_t1'] * data3['def_t1']
data6['ingr_t2_r'] = data6['ingr_t2'] * data6['def_t2']
data9['ingr_t3_r'] = data9['ingr_t3'] * data9['def_t3']
data12['ingr_t4_r'] = data12['ingr_t4'] * data12['def_t4']

/tmp/ipykernel_84268/2991205989.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ingr_t1_r'] = data3['ingr_t1'] * data3['def_t1']


In [44]:
print(data3['ingr_t1_r'].mean())
print(data6['ingr_t2_r'].mean())
print(data9['ingr_t3_r'].mean())
print(data12['ingr_t4_r'].mean())

441.42046757714786
443.3250086609635
512.7127088558354
451.78282074726246


## Regiones

In [36]:
# Corregimos los códigos para usarlos cómo texto
data3['ciudad'] = data3['ciudad'].apply(str)
data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == (fac3+4) else x)

data3['ciudad_2'] = data3['ciudad'].apply(lambda x: x[:2])

data6['ciudad'] = data6['ciudad'].apply(str)
data6['ciudad'] = data6['ciudad'].apply(lambda x: '0' + x if len(x) == (fac6+4) else x)

data6['ciudad_2'] = data6['ciudad'].apply(lambda x: x[:2])

data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == (fac9-1) else x)

data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:2])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == (fac12) else x)

data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:2])

/tmp/ipykernel_84268/2153798299.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ciudad'] = data3['ciudad'].apply(str)
/tmp/ipykernel_84268/2153798299.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ciudad'] = data3['ciudad'].apply(lambda x: '0' + x if len(x) == (fac3+4) else x)
/tmp/ipykernel_84268/2153798299.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the d

In [37]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [38]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
data3['region'] = data3['ciudad_2'].map(codigo_region)

data6['region'] = data6['ciudad_2'].map(codigo_region)

data9['region'] = data9['ciudad_2'].map(codigo_region)

data12['region'] = data12['ciudad_2'].map(codigo_region)

/tmp/ipykernel_84268/1690878332.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['region'] = data3['ciudad_2'].map(codigo_region)


In [39]:
data3['region'].value_counts()

region
Sierra                  6052
Guayas                  3619
Manabí                  3111
Pichincha               1608
Costa, Santo Domingo    1376
Los Ríos                1364
Azuay                   1181
El Oro                  1055
Amazonía                 819
Galápagos                 16
Name: count, dtype: int64

In [40]:
data6['region'].value_counts()

region
Sierra                  6089
Guayas                  3186
Manabí                  2973
Pichincha               1678
Costa, Santo Domingo    1357
Azuay                   1219
Los Ríos                1209
El Oro                  1068
Amazonía                1013
Galápagos                 35
Name: count, dtype: int64

In [41]:
data9['region'].value_counts()

region
Sierra                  14531
Guayas                  11042
Pichincha                7782
Amazonía                 5150
Azuay                    5079
Manabí                   4658
Costa, Santo Domingo     4451
El Oro                   4164
Los Ríos                 2741
Galápagos                 467
Name: count, dtype: int64

In [42]:
data12['region'].value_counts()

region
Sierra                  14329
Guayas                  10767
Pichincha                7744
Amazonía                 5206
Azuay                    4897
Manabí                   4595
Costa, Santo Domingo     4455
El Oro                   4102
Los Ríos                 2633
Galápagos                 480
Name: count, dtype: int64

## Calculo ingreso de los hogares

In [45]:
columnas_idef = pd.Index(['area', 'ciudad', 'zona', 'sector', 'vivienda',
       'hogar'])

data3['idef_hogar'] = data3['id_hogar']
data6['idef_hogar'] = data6['id_hogar']
data9['idef_hogar'] = data9['id_hogar']
data12['idef_hogar'] = data12['id_hogar']

print(len(data3['idef_hogar'].unique()))
print(len(data6['idef_hogar'].unique()))
print(len(data9['idef_hogar'].unique()))
print(len(data12['idef_hogar'].unique()))

16982
16980
16998
17001


/tmp/ipykernel_84268/1932811201.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['idef_hogar'] = data3['id_hogar']


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [46]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [47]:
data3['ingr_t1_h'] = data3.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data6['ingr_t2_h'] = data6.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data9['ingr_t3_h'] = data9.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data12['ingr_t4_h'] = data12.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

/tmp/ipykernel_84268/3738997912.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['ingr_t1_h'] = data3.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)


In [48]:
print(data3['ingr_t1_h'].mean())
print(data6['ingr_t2_h'].mean())
print(data9['ingr_t3_h'].mean())
print(data12['ingr_t4_h'].mean())

651.28838282909
656.1047481436425
754.7437434851371
661.2930278580577


## Sacamos edades negativas y mayores a 100 años

In [49]:
print(len(data3))
print(len(data6))
print(len(data9))
print(len(data12))

60173
60417
60065
59208


Transformamos las variables de edad a numericas para evitar problemas

In [50]:
data3['edad'] = pd.to_numeric(data3['p03'], errors='coerce')
data6['edad'] = pd.to_numeric(data6['p03'], errors='coerce')
data9['edad'] = pd.to_numeric(data9['p03'], errors='coerce')
data12['edad'] = pd.to_numeric(data12['p03'], errors='coerce')

/tmp/ipykernel_84268/1414182130.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['edad'] = pd.to_numeric(data3['p03'], errors='coerce')


In [51]:
data3 = data3.loc[(data3['edad'] >= 0) & (data3['edad'] < 100)]
data6 = data6.loc[(data6['edad'] >= 0) & (data6['edad'] < 100)]
data9 = data9.loc[(data9['edad'] >= 0) & (data9['edad'] < 100)]
data12 = data12.loc[(data12['edad'] >= 0) & (data12['edad'] < 100)]

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [52]:
k = 0.4
s = 0.9

In [53]:
# Si es necesario calcular el número de niños
data3['es_nino'] = data3['edad'] < 10
data3['ninos'] = data3.groupby('idef_hogar')['es_nino'].transform('sum')

data6['es_nino'] = data6['edad'] < 10
data6['ninos'] = data6.groupby('idef_hogar')['es_nino'].transform('sum')

data9['es_nino'] = data9['edad'] < 10
data9['ninos'] = data9.groupby('idef_hogar')['es_nino'].transform('sum')

data12['es_nino'] = data12['edad'] < 10
data12['ninos'] = data12.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data3['es_adulto'] = data3['edad'] > 10
data3['adultos'] = data3.groupby('idef_hogar')['es_adulto'].transform('sum')

data6['es_adulto'] = data6['edad'] > 10
data6['adultos'] = data6.groupby('idef_hogar')['es_adulto'].transform('sum')

data9['es_adulto'] = data9['edad'] > 10
data9['adultos'] = data9.groupby('idef_hogar')['es_adulto'].transform('sum')

data12['es_adulto'] = data12['edad'] > 10
data12['adultos'] = data12.groupby('idef_hogar')['es_adulto'].transform('sum')

In [54]:
data3['escala'] = (data3['adultos'] + k * data3['ninos']) ** s
data6['escala'] = (data6['adultos'] + k * data6['ninos']) ** s
data9['escala'] = (data9['adultos'] + k * data9['ninos']) ** s
data12['escala'] = (data12['adultos'] + k * data12['ninos']) ** s

In [55]:
data3['ingr_t_t1'] = data3['ingr_t1_h'] / data3['escala']
data6['ingr_t_t2'] = data6['ingr_t2_h'] / data6['escala']
data9['ingr_t_t3'] = data9['ingr_t3_h'] / data9['escala']
data12['ingr_t_t4'] = data12['ingr_t4_h'] / data12['escala']

In [56]:
print(data3['ingr_t_t1'].mean())
print(data6['ingr_t_t2'].mean())
print(data9['ingr_t_t3'].mean())
print(data12['ingr_t_t4'].mean())

197.90059225885418
198.14954086222252
230.45212304916777
203.1772955837409


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [57]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 2019

In [59]:
umbral_dict

{1: 78.7032739397182,
 2: 79.5614926013539,
 3: 80.1589598599,
 4: 80.8245161035507}

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [60]:
resultados_list = []

# Para cada trimeste
for t in [1, 2, 3, 4]:
    col_ingr = f'ingr_t_t{t}'
    umbral = umbral_dict.get(t)
    salario = salario_dict.get(t)
    
    # Selecciona el dataframe correspondiente
    if t == 1:
        df_actual = data3
    elif t == 2:
        df_actual = data6
    elif t == 3:
        df_actual = data9
    elif t == 4:
        df_actual = data12
    else:
        continue

    # Agrupa por región
    grouped = df_actual.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': t,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana + 1 if mediana < 1 else mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / (mediana + 1 if mediana < 1 else mediana)         
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2019,1,Amazonía,0.145268,0.066348,0.041231,0.073938,0.146920,0.226932,230.389942,171.466060,394.0,2.297831
1,2019,1,Azuay,0.143229,0.066927,0.045882,0.056894,0.118710,0.203661,182.629083,156.421323,394.0,2.518838
2,2019,1,"Costa, Santo Domingo",0.276339,0.103853,0.058196,0.068654,0.134490,0.210390,143.448215,112.832617,394.0,3.491898
3,2019,1,El Oro,0.101424,0.035722,0.018028,0.062544,0.121004,0.178627,220.079896,168.637848,394.0,2.336368
4,2019,1,Galápagos,0.157676,0.157676,0.157676,0.167491,0.337424,0.574431,188.022630,87.286779,394.0,4.513857
5,2019,1,Guayas,0.211655,0.084960,0.047503,0.097255,0.183918,0.266711,201.558814,135.734187,394.0,2.902732
6,2019,1,Los Ríos,0.216475,0.086066,0.049502,0.060590,0.118323,0.177383,151.826157,125.100489,394.0,3.149468
7,2019,1,Manabí,0.213676,0.070748,0.035175,0.060601,0.117956,0.177910,161.495546,131.651714,394.0,2.992745
8,2019,1,Pichincha,0.181243,0.082531,0.057791,0.106852,0.207318,0.327722,235.774159,155.030737,394.0,2.541432
9,2019,1,Sierra,0.148583,0.061438,0.036957,0.082902,0.161154,0.242254,236.496930,169.066441,394.0,2.330445


### Inserta los cálculos en la base final

In [61]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [62]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_final_regional.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_final_regional.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')